# Kanonische Becken-Konsistenz: PedRecNet v2

Misst auf **denselben Validierungsbildern** für neue und archivierte v2-Gewichte:

\[
\Delta_{pelvis}=p_{hip\_center}-\frac{p_{left\_hip}+p_{right\_hip}}{2}.
\]

Die kanonische SIM-Konvention definiert die Wurzel als Hüftmittelpunkt. Ein großes
$\|\Delta_{pelvis}\|$ ist daher eine **interne geometrische Inkonsistenz**, auch ohne GT.
Ein kleines Delta beweist umgekehrt weder richtige Körperproportionen noch gute Pose.

Das Notebook führt **den nackten Inferenz-Forward ohne Adapter und Loss** aus.
So bleiben alte v2-Gewichte trotz geänderter Adapter-Maske vergleichbar.
Dataset-GT kann andere Konventionen haben; dessen Becken-Delta wird separat gezeigt.
Die Konventionen von 2D und 3D dürfen nicht aus einem gemeinsamen Zahlenwert abgeleitet werden.

Auf dem **GPU-Host** von oben nach unten ausführen. Alle Ergebniszellen sind zunächst leer.
Keine vollständige Validierung auf der CPU; ein expliziter CPU-Smoke-Modus erlaubt maximal acht Bilder pro Set.
Benötigt die Projektumgebung plus Jupyter/IPython-Kernel (bei Bedarf
`uv pip install notebook ipykernel`; Kernel mit der Projekt-`.venv` wählen).

In [ ]:
from pathlib import Path
import os
import sys

ROOT_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "pedrec").is_dir() and (p / "mise.toml").is_file()), None)
if ROOT_DIR is None:
    raise RuntimeError("Notebook aus dem Repository oder dessen notebooks/ starten.")
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import hashlib
import json
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from pedrec.configs.pedrec_net_config import PedRecNetConfig
from pedrec.models.constants.skeleton_pedrec import SKELETON_PEDREC_PARENT_CHILD_PAIRS
from pedrec.networks.net_pedrec.pedrec_net import PedRecNet
from pedrec.training.train_pedrec import get_experiment_description, IMAGENET_TRANSFORM
from pedrec.training.experiments.pedrec_stages import get_stage
from pedrec.training.experiments.experiment_path_helper import get_experiment_paths
from pedrec.training.experiments.experiment_dataset_helper import get_validation_sets
from pedrec.utils.torch_utils.checkpoint_io import load_state_dict_file
from pedrec.tools.experiments.pelvis_consistency import (
    ROOT, LEFT, RIGHT, collect_samples, summarize_pelvis, summarize_orientation,
)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.2})
pd.set_option("display.max_columns", 30)

## Konfiguration

`DATA_ROOT` enthält `datasets/` und `models/`. Die Auswahl liegt bereits auf den
**Trainings-Validierungssplits** (H36M jedes 64. Bild, SIM nur Holdout-Avatare).
`MAX_SAMPLES_PER_SET` zieht daraus eine feste Zufallsstichprobe; `None` nimmt das volle Set.
Vergleich alt/neu ist gepaart, aber viele Frames einer Person sind statistisch abhängig.
Schwellen von 10/30/100 mm sind Diagnosehilfen, keine etablierten Qualitätsgrenzen.

Zum späteren Auswerten ohne erneute Inferenz: `RUN_INFERENCE = False`, gleichen `OUTPUT_DIR` verwenden.
Ein erneuter Inferenzlauf ersetzt die Diagnose-Dateien in diesem Ordner; Gewichte und Trainingsprotokolle bleiben erhalten.

In [ ]:
DATA_ROOT = Path(os.environ.get("PEDREC_DATA_DIR", ROOT_DIR / "data")).expanduser().resolve()
RESULT_DIR = DATA_ROOT / "models/pedrec/single_results"
CHECKPOINTS = {
    "v2_2026-10-30_best": RESULT_DIR / "experiment_pedrec_v2_p2d3d_c_o_0_best.pth",
    "v2_2026-10-28_best": RESULT_DIR / "archive-2026-10-28/experiment_pedrec_v2_p2d3d_c_o_0_best.pth",
    # Optional: finalen neuen Checkpoint ergänzen.
    # "v2_2026-10-30_final": RESULT_DIR / "experiment_pedrec_v2_p2d3d_c_o_0.pth",
}
DATASETS = ["SIM", "H36M", "Fit3D", "AIST++", "COCO"]
MAX_SAMPLES_PER_SET = 1024
SEED = 42
BATCH_SIZE = 32
NUM_WORKERS = 0  # robust im Notebook, bei Bedarf auf dem Host erhöhen
KEEP_EXAMPLES_PER_SET = 6
ALLOW_CPU_SMOKE = False
RUN_INFERENCE = True
OUTPUT_DIR = DATA_ROOT / "models/pedrec/pelvis_analysis/v2_2026-10-30-31"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Daten:", DATA_ROOT)
print("Gerät:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
print("Ergebnisse:", OUTPUT_DIR)
if RUN_INFERENCE:
    if DEVICE.type != "cuda" and not (
        ALLOW_CPU_SMOKE and MAX_SAMPLES_PER_SET is not None and 1 <= MAX_SAMPLES_PER_SET <= 8
    ):
        raise RuntimeError("Kein CUDA. Auf GPU-Host starten oder explizit maximal 8 CPU-Smoke-Bilder/Set wählen.")
    missing = [str(p) for p in CHECKPOINTS.values() if not p.is_file()]
    if missing:
        raise FileNotFoundError("Checkpoint fehlt: " + ", ".join(missing))
    if MAX_SAMPLES_PER_SET is not None and MAX_SAMPLES_PER_SET < 1:
        raise ValueError("MAX_SAMPLES_PER_SET muss positiv oder None sein.")

## Datensätze und identische Bildauswahl

Die vorhandenen Dataset-Klassen übernehmen Crop, RGB-Normalisierung und GT-Flags.
Keine Trainings-Augmentation, kein Flip-Test und keine Ausrichtung an GT bei der Inferenz.
COCO besitzt keine 3D-GT: Konsistenz ist dort messbar, 3D-Genauigkeit nicht.
Die 3D-Skalierung entspricht der Dataset-Konfiguration (aktuell 3000 mm).

In [ ]:
selected_sets, sample_indices = {}, {}
if RUN_INFERENCE:
    stage = get_stage("p2d3d_c_o")
    description = get_experiment_description(stage, get_experiment_paths(str(DATA_ROOT)),
                                             batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
    description.use_val_coco = "COCO" in DATASETS
    description.use_val_sim = "SIM" in DATASETS
    description.use_val_h36m = "H36M" in DATASETS
    description.use_extra_3d = bool(set(DATASETS) & {"MPI-INF-3DHP", "Fit3D", "AIST++"})
    available_sets = {s.name: s for s in get_validation_sets(description, IMAGENET_TRANSFORM)}
    missing = set(DATASETS) - available_sets.keys()
    if missing:
        raise RuntimeError(f"Angeforderte Sets fehlen: {sorted(missing)}; DATASETS anpassen oder Daten bereitstellen.")
    for name in DATASETS:
        val_set = available_sets[name]
        n = len(val_set.loader.dataset)
        if not n:
            raise RuntimeError(f"Leeres Set: {name}")
        cap = n if MAX_SAMPLES_PER_SET is None else min(n, MAX_SAMPLES_PER_SET)
        indices = np.sort(np.random.default_rng(SEED).choice(n, cap, replace=False)).tolist()
        selected_sets[name], sample_indices[name] = val_set, indices
    display(pd.DataFrame([{"dataset": name, "available": len(s.loader.dataset),
                           "selected": len(sample_indices[name])} for name, s in selected_sets.items()]))

## Kanonische Vorhersagen sammeln und exportieren

Es werden nur `model.*`-Gewichte geladen, **strict** gegen das aktuelle v2-Inferenznetz.
Die geänderte Adapter-Maske ist hier irrelevant; v1-Gewichte passen dagegen nicht in dieses Netz.
Pro Set bleiben nur die schlechtesten Becken-Beispiele mit Crop und Koordinaten im Speicher.
CSV, Auswahl und Gewichts-Hashes erlauben die spätere Zuordnung und Wiederholung.

In [ ]:
def checkpoint_hash(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

examples = {}
if RUN_INFERENCE:
    torch.manual_seed(SEED)
    frames, provenance = [], []
    cfg = PedRecNetConfig()
    for tag, checkpoint in CHECKPOINTS.items():
        print(tag, checkpoint)
        state = load_state_dict_file(str(checkpoint))
        model = PedRecNet(cfg, pretrained_backbone=False)
        bare_state = {k.removeprefix("model."): v for k, v in state.items() if k.startswith("model.")}
        model.load_state_dict(bare_state, strict=True)
        del state, bare_state
        model.to(DEVICE).eval()
        provenance.append({"tag": tag, "path": str(checkpoint), "sha256": checkpoint_hash(checkpoint)})
        for name, val_set in selected_sets.items():
            range_mm = (val_set.val_set_cfg.skeleton_3d_range if val_set.val_set_cfg is not None
                        else description.sim_val_dataset_cfg.skeleton_3d_range)
            frame, kept = collect_samples(model, val_set.loader.dataset, sample_indices[name], DEVICE,
                                          cfg.model.input_size, range_mm, BATCH_SIZE, NUM_WORKERS,
                                          KEEP_EXAMPLES_PER_SET)
            frames.append(frame.assign(checkpoint=tag, dataset=name))
            examples[(tag, name)] = kept
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    samples = pd.concat(frames, ignore_index=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    samples.to_csv(OUTPUT_DIR / "samples.csv", index=False)
    manifest = []
    for (tag, dataset), kept in examples.items():
        for item in kept:
            filename = f"example_{len(manifest):04d}.npz"
            np.savez_compressed(OUTPUT_DIR / filename, **item)
            manifest.append({"checkpoint": tag, "dataset": dataset, "file": filename})
    (OUTPUT_DIR / "examples.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    metadata = {"created_utc": datetime.now(timezone.utc).isoformat(), "seed": SEED,
                "sample_indices": sample_indices, "checkpoints": provenance, "device": str(DEVICE),
                "max_samples_per_set": MAX_SAMPLES_PER_SET,
                "torch_version": torch.__version__, "prediction": "canonical; no adapter or loss",
                "error_aggregation": "per-image mean over visible supported joints excluding root"}
    (OUTPUT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
else:
    samples = pd.read_csv(OUTPUT_DIR / "samples.csv")
    for entry in json.loads((OUTPUT_DIR / "examples.json").read_text(encoding="utf-8")):
        with np.load(OUTPUT_DIR / entry["file"], allow_pickle=False) as archive:
            item = {k: archive[k].copy() for k in archive.files}
        for key in ("sample_index", "pelvis_3d_mm", "image_path"):
            item[key] = item[key].item()
        examples.setdefault((entry["checkpoint"], entry["dataset"]), []).append(item)

pelvis_summary = summarize_pelvis(samples)
orientation_summary = summarize_orientation(samples)
if RUN_INFERENCE:
    pelvis_summary.to_csv(OUTPUT_DIR / "pelvis_summary.csv", index=False)
    orientation_summary.to_csv(OUTPUT_DIR / "orientation_summary.csv", index=False)
print(f"{len(samples)} Vorhersagen; {OUTPUT_DIR / 'samples.csv'}")

## Beckenfehler: Betrag, Richtung, lange Verteilungsschwänze

`pelvis_d*_mm` ist **Wurzel minus Hüftmittelpunkt in Kameraachsen**, 3D-y zeigt nach oben.
Vorzeichen können sich über verschiedene Ansichten aufheben; Norm und P95 sind deshalb zentral.
`gt_pelvis_3d_mm_median` ist eine Kontrolle der jeweiligen Annotation, keine kanonische Zielvorgabe.
2D-Abstand in Pixeln gilt im 192×256-Netzeingang, nicht im Originalbild.

In [ ]:
display(pelvis_summary.round(2))

names = samples.dataset.unique()
fig, axes = plt.subplots(1, len(names), figsize=(4 * len(names), 3.8), squeeze=False)
for ax, name in zip(axes[0], names):
    for tag, group in samples[samples.dataset == name].groupby("checkpoint", sort=False):
        values = np.sort(group.pelvis_3d_mm.dropna().to_numpy())
        if len(values):
            ax.plot(values, np.arange(1, len(values) + 1) / len(values), label=tag)
    ax.set(title=name, xlabel="Kanonischer Beckenfehler [mm]", ylabel="Anteil ≤ x", ylim=(0, 1.02))
    ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "pelvis_ecdf.png", dpi=160)
plt.show()

## Alt gegen neu auf identischen Bildern

Delta = neuer minus alter Beckenfehler; negativ ist besser. Das ist ein gepaarter
Vergleich der internen Konsistenz, kein Nachweis höherer 3D-Genauigkeit.
Es werden keine Konfidenzintervalle über vermeintlich unabhängige Videoframes berechnet.

In [ ]:
REFERENCE = "v2_2026-10-28_best"
CANDIDATE = "v2_2026-10-30_best"
key = ["dataset", "sample_index"]
old = samples[samples.checkpoint == REFERENCE]
new = samples[samples.checkpoint == CANDIDATE]
if len(old) and len(new):
    paired = new.merge(old, on=key, suffixes=("_new", "_old"), validate="one_to_one")
    if len(paired) != len(old) or len(paired) != len(new):
        raise RuntimeError("Alt/neu haben unterschiedliche Bildauswahl.")
    assert (paired.image_path_new == paired.image_path_old).all()
    paired["delta_mm"] = paired.pelvis_3d_mm_new - paired.pelvis_3d_mm_old
    display(paired.groupby("dataset").agg(
        n=("delta_mm", "count"), mean_delta_mm=("delta_mm", "mean"),
        median_delta_mm=("delta_mm", "median"),
        improved_pct=("delta_mm", lambda v: 100 * (v.dropna() < 0).mean()),
    ).round(2))
else:
    print("Für gepaarten Vergleich beide CHECKPOINTS aktivieren.")

## Ist ein großer Teil des Fehlers eine Verschiebung gegen die Wurzel?

Beide Fehler beziehen sich auf die **kanonische Vorhersage und Dataset-GT** mit
verbleibenden Konventionsunterschieden. Vergleich A zentriert beide Skelette an ihrer
eigenen Wurzel; Vergleich B an ihrem eigenen Hüftmittelpunkt. In beiden Fällen wird
`hip_center` aus den ausgewerteten Gelenken ausgeschlossen. Sichtbarkeit und Unterstützung
müssen vorliegen. Eine größere Verbesserung durch B passt zu einer Inkonsistenz,
beweist aber keine alleinige Ursache. B ist eine Diagnose, **keine neue Benchmarkzahl**.
Die Mittelwerte pro Bild unterscheiden sich von der gelenkweisen Mittelung im Trainingsprotokoll.

In [ ]:
accuracy = samples.dropna(subset=["root_relative_error_mm", "hip_relative_error_mm"]).copy()
accuracy["centering_gain_mm"] = accuracy.root_relative_error_mm - accuracy.hip_relative_error_mm
if len(accuracy):
    display(accuracy.groupby(["checkpoint", "dataset"], sort=False)[
        ["root_relative_error_mm", "hip_relative_error_mm", "centering_gain_mm"]
    ].median().round(2))
    fig, ax = plt.subplots(figsize=(7, 4))
    for (tag, name), group in accuracy.groupby(["checkpoint", "dataset"], sort=False):
        ax.scatter(group.pelvis_3d_mm, group.centering_gain_mm, s=8, alpha=0.25, label=f"{tag}: {name}")
    ax.axhline(0, color="black", lw=1)
    ax.set(xlabel="Kanonischer Beckenfehler [mm]", ylabel="Fehler Wurzel − Fehler Hüftmittel [mm]")
    ax.legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "pelvis_vs_centering.png", dpi=160)
    plt.show()

## Auffällige Bilder und unverändertes kanonisches Skelett

Rot: vorhergesagte Wurzel; Magenta: Mittelwert der vorhergesagten Hüften.
Die Verbindung macht den Widerspruch sichtbar. Die 3D-Darstellung verschiebt lediglich
die **gesamte** Pose zur Wurzel, ohne eine inkonsistente Hüftlage zu korrigieren.
Der 2D-Head ist unabhängig vom 3D-Head; dessen Punkte im Crop sind keine Kameraprojektion der 3D-Pose.

In [ ]:
def show_example(item, title=""):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    rgb = np.clip(item["crop"].transpose(1, 2, 0) * std + mean, 0, 1)
    p2 = item["pose_2d_px"]
    p3 = item["pose_3d_mm"] - item["pose_3d_mm"][ROOT]
    fig = plt.figure(figsize=(10, 4))
    ax = fig.add_subplot(121)
    ax.imshow(rgb)
    for a, b in SKELETON_PEDREC_PARENT_CHILD_PAIRS:
        ax.plot(p2[[a, b], 0], p2[[a, b], 1], lw=0.8, color="cyan", alpha=0.7)
    mid2 = p2[[LEFT, RIGHT]].mean(axis=0)
    ax.scatter(*p2[ROOT], color="red", s=45, label="root")
    ax.scatter(*mid2, color="magenta", marker="x", s=70, label="hip mean")
    ax.plot([p2[ROOT, 0], mid2[0]], [p2[ROOT, 1], mid2[1]], color="magenta", lw=2)
    ax.set(xlim=(0, rgb.shape[1] - 1), ylim=(rgb.shape[0] - 1, 0), title="2D-Head im Crop")
    ax.legend(fontsize=8)
    ax = fig.add_subplot(122, projection="3d")
    # Display camera x, depth z, up y, preserving the actual coordinates.
    display_pose = p3[:, [0, 2, 1]]
    for a, b in SKELETON_PEDREC_PARENT_CHILD_PAIRS:
        ax.plot(*display_pose[[a, b]].T, color="steelblue", lw=1)
    mid3 = display_pose[[LEFT, RIGHT]].mean(axis=0)
    ax.scatter(*display_pose[ROOT], color="red", s=45)
    ax.scatter(*mid3, color="magenta", marker="x", s=70)
    ax.plot(*np.stack([display_pose[ROOT], mid3]).T, color="magenta", lw=3)
    ax.set(xlabel="Kamera x [mm]", ylabel="Tiefe z [mm]", zlabel="Kamera y [mm]", title="Kanonischer 3D-Head")
    ax.set_box_aspect(np.maximum(np.ptp(display_pose, axis=0), 1))
    fig.suptitle(f"{title} · Sample {item['sample_index']} · Δ={item['pelvis_3d_mm']:.1f} mm")
    fig.tight_layout()
    plt.show()
    print(item["image_path"])

EXAMPLE_CHECKPOINT = "v2_2026-10-30_best"
EXAMPLE_DATASET = "H36M"
for item in examples.get((EXAMPLE_CHECKPOINT, EXAMPLE_DATASET), [])[:3]:
    show_example(item, f"{EXAMPLE_CHECKPOINT} / {EXAMPLE_DATASET}")

## Orientierung: Körper und Kopf getrennt, Winkelumlauf korrekt

Diese Werte entstehen ohne weiteren Forward auf denselben Bildern. Fehler ist der
kürzeste Azimut-Abstand auf dem Kreis, Wertebereich 0–180°. Fehlende Labels bleiben NaN.
Median, P95, Acc22.5 und grobe Fehler >90° ergänzen den Mittelwert. COCO liefert Körper-,
SIM auch Kopf-Labels; andere Sets liefern hier gewöhnlich keine Orientierung.

**Kein direkter Vergleich mit den alten v1-SIM-Zahlen:** v1 hatte Trainingsdaten im
SIM-Validierungsset. Für eine Head-Ablation müssen Daten, Backbone, Pose-Eingabe,
Trainingsbudget und Seeds gleich sein. v1 ist hier nicht geladen; die Referenzwerte
stehen in `doc/experiments/README.md` und im Architektur-Fazit.

In [ ]:
display(orientation_summary.round(2))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, part in zip(axes, ("body", "head")):
    for (tag, name), group in samples.groupby(["checkpoint", "dataset"], sort=False):
        values = group[f"{part}_phi_error_deg"].dropna()
        if len(values):
            ax.hist(values, bins=np.linspace(0, 180, 37), density=True,
                    histtype="step", label=f"{tag}: {name}")
    ax.set(title=part, xlabel="Phi-Fehler [Grad]", ylabel="Dichte")
    if ax.get_legend_handles_labels()[0]:
        ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "orientation_errors.png", dpi=160)
plt.show()

# Ausreißer nach tatsächlicher Blickrichtung: kleine Zellen anhand n erkennen.
by_angle = []
for part in ("body", "head"):
    current = samples.dropna(subset=[f"{part}_phi_error_deg"]).copy()
    if current.empty:
        continue
    current["azimuth_bin"] = pd.cut(current[f"{part}_phi_gt_deg"] % 360,
                                     bins=np.arange(0, 361, 45), right=False)
    table = current.groupby(["checkpoint", "dataset", "azimuth_bin"], observed=True)[f"{part}_phi_error_deg"].agg(
        n="count", mean_deg="mean", p95_deg=lambda s: s.quantile(0.95),
    ).reset_index()
    by_angle.append(table.assign(part=part))
if by_angle:
    display(pd.concat(by_angle, ignore_index=True).round(2))

## Interpretation und nächster Versuch

- **Neu deutlich kleiner als alt, besonders auf H36M/Fit3D:** Der Fix verbessert
  die kanonische interne Konsistenz. Ein kleineres Dataset-MPJPE ist davon getrennt zu prüfen.
- **SIM klein, reale Sets groß:** Hinweis auf eine domänenabhängige Abweichung.
  Crop-Beispiele und Vorzeichen prüfen; nicht allein aus der Adapter-Lücke folgern.
- **Auch SIM groß:** Die explizite Wurzel und die vorhergesagten Hüften sind bereits
  im kanonischen Trainingsbereich inkonsistent. Eine strukturelle Bindung der Wurzel
  an den Hüftmittelpunkt ist dann ein plausibler kontrollierter Versuch.
- **Konsistenz gut, Genauigkeit schlecht:** Tiefenambiguität, Proportionen und übrige
  Gelenkkonventionen bleiben mögliche Ursachen. Diese Metrik erklärt nicht alles.
- **Orientierung:** Schlechtere Tails oder Front/Back-Verwechslungen können trotz gutem
  Mittelwert auftreten. Biternion gegen zirkuläre Bin-Verteilung unter sonst gleichen
  Bedingungen testen; der lineare Erwartungswert über 0–360° ist keine saubere zirkuläre Referenz.

Eine deterministisch aus den Hüften abgeleitete Wurzel würde diese Metrik automatisch
auf null setzen. Das allein wäre noch kein Qualitätsgewinn: immer zusätzlich 3D-Fehler,
Proportionen und Ausgaben auf realen Bildern prüfen. Keine automatische Gewichtsänderung
oder nachträgliche Reparatur der Pose in diesem Notebook.